**Tools**


Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [2]:
import os 
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("openai/gpt-oss-20b", model_provider="groq")
response = model.invoke("Hello how are you?")
response.text

'Hello! I’m doing great—thanks for asking. How can I help you today?'

In [3]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [4]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'We need to call get_weather.', 'tool_calls': [{'id': 'fc_e5199caa-ad9a-404e-9c95-aae81d08f300', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 127, 'total_tokens': 158, 'completion_time': 0.033937642, 'completion_tokens_details': {'reasoning_tokens': 8}, 'prompt_time': 0.006252092, 'prompt_tokens_details': None, 'queue_time': 0.34181998, 'total_time': 0.040189734}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_63473661d7', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a167a7-aa63-7053-ba5e-b76b42121c3d-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_e5199caa-ad9a-404e-9c95-aae81d08f300', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 127, 'output_tokens': 31, 'to

In [5]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

It’s sunny in Boston.


In [6]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call the get_weather function.', 'tool_calls': [{'id': 'fc_7cd672f4-ba39-4aa0-a3f9-d819f42a4bd9', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 126, 'total_tokens': 159, 'completion_time': 0.035255989, 'completion_tokens_details': {'reasoning_tokens': 10}, 'prompt_time': 0.00850891, 'prompt_tokens_details': None, 'queue_time': 0.210253025, 'total_time': 0.043764899}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a8c584dda7', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a167a7-ac50-7629-ba10-9d2c1f4c9158-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_7cd672f4-ba39-4aa0-a3f9-d819f42a4bd9', 'type': 'tool